# 1、HumanInTheLoopMiddleware中间件

# 工具调用的中断

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.middleware import SummarizationMiddleware

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
    profile={"max_input_tokens": 1000000}
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断了..."
                },
            },
            description_prefix="中断啦！！"
        ),
    ]
)

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({
    "messages": [HumanMessage(content="请帮我查询今天北京的天气"
                                      "查询今日新闻"
                                      "查看ID为 'sk2131421' 的邮件内容，"
                                      "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                                      "同时做这四件事")]
},
    config=config
)


rprint(response)

{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='96f73f16-9e79-4866-a53c-99b0bfc694dd'
        ),
        AIMessage(
            content='好的！这四件事互相独立，我同时帮您处理！🚀',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 170,
                    'prompt_tokens': 391,
                    'total_tokens': 561,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 83,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.2',
                'system_fingerprint': None,
                'id': '20260713140431f1297030fefc42f6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f5a13-bf56-74d1-8ad4-ca5fe2a0a156-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'id': 'call_-7453169792288024252',
                    'type': 'tool_call'
                },
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_-7453169792288024251',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_-7453169792288024250',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_-7453169792288024249', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 391,
                'output_tokens': 170,
                'total_tokens': 561,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 83}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': False},
                        'description': "中断啦！！\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': 
False}"
                    },
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断了...'
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦！！\n\nTool: get_news\nArgs: {}'}
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']},
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']}
                ]
            },
            id='21492c66fc9f86caa4a779756d4498dd'
        )
    ]
}

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError(\'HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Max retries exceeded with url: /info (Caused by ReadTimeoutError("HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Read timed out. (read timeout=10.0)"))\'))\nContent-Length: None\nAPI Key: lsv2_********************************************cf')
Run compression is not enabled. Please update to the latest version of LangSmith. Falling back to regular multipart ingestion.


In [3]:
weather_decision = {
    "type" : "edit",
    "edited_action" : {
        "name" : "get_weather",
        "args" : {"city" : "上海市","is_forcast" : True},
    }
}


news_decision = {
    "type" : "approve"
}


send_email_decision = {
    "type" : "approve"
}



decisions = {
    "decisions" : []
}

interrupts = response.get("__interrupt__",[])
action_requests = interrupts[0].value["action_requests"]


for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts :
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config = config
    )

    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

好的！这四件事互相独立，我同时帮您处理！🚀
Tool Calls:
  get_weather (call_-7453169792288024252)
 Call ID: call_-7453169792288024252
  Args:
    city: 上海市
    is_forcast: True
  read_email_tool (call_-7453169792288024251)
 Call ID: call_-7453169792288024251
  Args:
    email_id: sk2131421
  send_email_tool (call_-7453169792288024250)
 Call ID: call_-7453169792288024250
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
  get_news (call_-7453169792288024249)
 Call ID: call_-7453169792288024249
  Args:
================================= Tool Message =================================
Name: get_weather

上海市今天天气不错
明天下雨
================================= Tool Message =================================
Name: read_email_tool

邮件